# BigAlpha 2026 AI 因子挖掘提交版

这个 Notebook 是比赛提交版，核心入口是 `main(data_source, start_datetime, end_datetime)`。

返回结果必须且仅包含三列：`date`, `instrument`, `factor`。

核心逻辑：

1. 用 BigQuant DAI 从 `bigalpha_2026_stock_bar1m` 读取 1 分钟 K 线与一档盘口数据；
2. 聚合为日频特征；
3. 构造微观结构、反转、动量、流动性、风险等信号；
4. `# [AI-CORE]` 使用滚动 Ridge 模型训练历史特征对下一日截面收益的预测；
5. 返回日频因子。


In [1]:
"""BigAlpha 2026 AI Factor Mining submission template.

This file is designed for the BigQuant competition environment.
The evaluator imports `main` and calls it with a data source name, start datetime,
and end datetime. The returned DataFrame must contain exactly three columns:
`date`, `instrument`, `factor`.

Core idea:
- Use only the allowed BigAlpha minute bar / order-book data source.
- Aggregate 1-minute data to daily stock-level features with DAI.
- Build interpretable microstructure signals: intraday reversal, order-book imbalance,
  spread pressure, liquidity, volatility and momentum.
- [AI-CORE] Train a rolling Ridge model on historical features to predict next-day
  cross-sectional returns. For each output date, the model only uses observations
  strictly before that date, avoiding look-ahead bias.
"""

from __future__ import annotations

import warnings
from typing import Any

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

EPS = 1e-8
DEFAULT_BAR_TABLE = "bigalpha_2026_stock_bar1m"
MIN_TRAIN_ROWS = 2_000
MAX_TRAIN_ROWS = 180_000
RETRAIN_EVERY_DAYS = 28
LOOKBACK_DAYS = 760
RANDOM_SEED = 42


def _resolve_bar_table(data_source: Any) -> str:
    """Resolve the minute-bar table name from the platform input."""
    if isinstance(data_source, str) and data_source.strip():
        return data_source.strip()
    if isinstance(data_source, dict):
        for key in ["bar1m", "stock_bar1m", "minute_bar", "data_source", "table"]:
            value = data_source.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()
    return DEFAULT_BAR_TABLE


def _to_timestamp(value: Any, default: str) -> pd.Timestamp:
    if value is None:
        return pd.Timestamp(default)
    ts = pd.to_datetime(value)
    if pd.isna(ts):
        return pd.Timestamp(default)
    return pd.Timestamp(ts)


def _fmt_filter_datetime(ts: pd.Timestamp, end_of_day: bool = False) -> str:
    ts = pd.Timestamp(ts)
    if end_of_day:
        ts = ts.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)
    else:
        ts = ts.normalize()
    return ts.strftime("%Y-%m-%d %H:%M:%S")


def _safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    return a.astype(float) / b.astype(float).replace(0, np.nan)


def _cs_zscore(frame: pd.DataFrame, col: str) -> pd.Series:
    """Cross-sectional z-score by date; robust to all-NaN / constant groups."""
    def z(s: pd.Series) -> pd.Series:
        s = pd.to_numeric(s, errors="coerce")
        med = s.median()
        s = s.fillna(med)
        std = s.std(ddof=0)
        if not np.isfinite(std) or std < EPS:
            return pd.Series(0.0, index=s.index)
        return ((s - s.mean()) / (std + EPS)).clip(-5, 5)

    return frame.groupby("date", group_keys=False)[col].apply(z)


def _load_daily_features_from_dai(
    bar_table: str,
    query_start: pd.Timestamp,
    query_end: pd.Timestamp,
) -> pd.DataFrame:
    """Aggregate BigAlpha 1-minute bar and level-1 order-book data to daily features."""
    import dai  # Imported inside the function because it only exists in BigQuant AIStudio.

    start_s = _fmt_filter_datetime(query_start, end_of_day=False)
    end_s = _fmt_filter_datetime(query_end, end_of_day=True)

    # Keep the SQL deliberately simple and based on fields shown in the official data page.
    # Heavy operations are pushed to DAI; pandas receives only daily rows.
    sql = f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            LAST(pre_close) AS pre_close,
            MAX(high) AS high,
            MIN(low) AS low,
            LAST(close) AS close,
            LAST(volume) AS volume,
            LAST(amount) AS amount,
            AVG(close) AS avg_close,
            AVG(ask_price1) AS ask_price1_avg,
            AVG(bid_price1) AS bid_price1_avg,
            AVG(ask_volume1) AS ask_volume1_avg,
            AVG(bid_volume1) AS bid_volume1_avg
        FROM {bar_table}
        GROUP BY date::DATE, instrument
        ORDER BY date, instrument
    """
    data = dai.query(
        sql,
        filters={"date": [start_s, end_s]},
        compression=True,
    ).df()
    return data


def _build_daily_features(daily: pd.DataFrame) -> pd.DataFrame:
    """Create predictive daily features from daily aggregated minute data."""
    df = daily.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    numeric_cols = [
        "pre_close", "high", "low", "close", "volume", "amount", "avg_close",
        "ask_price1_avg", "bid_price1_avg", "ask_volume1_avg", "bid_volume1_avg",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    mid = (df["ask_price1_avg"] + df["bid_price1_avg"]) / 2.0
    vol_sum = df["ask_volume1_avg"] + df["bid_volume1_avg"]

    df["ret_1d"] = _safe_divide(df["close"], df["pre_close"]) - 1.0
    df["range_pct"] = _safe_divide(df["high"] - df["low"], df["pre_close"].abs())
    df["avg_to_close"] = _safe_divide(df["avg_close"], df["close"]) - 1.0
    df["spread_pct"] = _safe_divide(df["ask_price1_avg"] - df["bid_price1_avg"], mid.abs())
    df["order_imbalance"] = _safe_divide(df["bid_volume1_avg"] - df["ask_volume1_avg"], vol_sum.abs())
    df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
    df["volume_log"] = np.log1p(df["volume"].clip(lower=0))
    df["vwap"] = _safe_divide(df["amount"], df["volume"])
    df["close_to_vwap"] = _safe_divide(df["close"], df["vwap"]) - 1.0
    df["illiquidity"] = _safe_divide(df["ret_1d"].abs(), df["amount"].abs() + 1.0)

    g = df.groupby("instrument", group_keys=False)
    df["mom_5d"] = g["close"].pct_change(5)
    df["mom_20d"] = g["close"].pct_change(20)
    df["mom_60d"] = g["close"].pct_change(60)
    df["vol_20d"] = g["ret_1d"].transform(lambda s: s.rolling(20, min_periods=5).std())
    df["range_5d"] = g["range_pct"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["imbalance_5d"] = g["order_imbalance"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["spread_5d"] = g["spread_pct"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["avg_to_close_5d"] = g["avg_to_close"].transform(lambda s: s.rolling(5, min_periods=3).mean())
    df["amount_z20"] = g["amount_log"].transform(
        lambda s: (s - s.rolling(20, min_periods=5).mean()) / (s.rolling(20, min_periods=5).std() + EPS)
    )

    # Target is only used for rolling historical model training.
    df["target_next_1d"] = g["close"].shift(-1) / df["close"] - 1.0
    df["target_z"] = _cs_zscore(df, "target_next_1d")
    return df.sort_values(["date", "instrument"]).reset_index(drop=True)


def _add_rule_factor(df: pd.DataFrame) -> pd.DataFrame:
    """Build an interpretable non-AI fallback factor."""
    out = df.copy()
    base_cols = [
        "avg_to_close_5d",      # intraday reversal: close below intraday average may rebound
        "imbalance_5d",         # bid-side depth pressure
        "close_to_vwap",        # price-vwap dislocation
        "amount_z20",           # abnormal liquidity participation
        "mom_20d",              # medium-term momentum
        "spread_5d",            # trading friction / crowding risk
        "vol_20d",              # realized risk
        "range_5d",             # intraday instability
        "illiquidity",          # liquidity risk
    ]
    for col in base_cols:
        out[col + "_z"] = _cs_zscore(out, col)

    # Direction: larger factor is better.
    out["rule_factor"] = (
        0.24 * out["avg_to_close_5d_z"]
        + 0.22 * out["imbalance_5d_z"]
        - 0.12 * out["close_to_vwap_z"]
        + 0.10 * out["amount_z20_z"]
        + 0.12 * out["mom_20d_z"]
        - 0.16 * out["spread_5d_z"]
        - 0.12 * out["vol_20d_z"]
        - 0.10 * out["range_5d_z"]
        - 0.06 * out["illiquidity_z"]
    )
    out["rule_factor"] = _cs_zscore(out, "rule_factor")
    return out


def _add_walk_forward_ai_factor(
    df: pd.DataFrame,
    start_dt: pd.Timestamp,
    end_dt: pd.Timestamp,
) -> pd.DataFrame:
    """[AI-CORE] Rolling model training and out-of-sample prediction.

    For each prediction date d, training rows are restricted to date < d. This
    avoids using the target of the prediction date or any future label.
    """
    out = df.copy()
    out["ai_model_pred"] = np.nan

    feature_cols = [
        "ret_1d", "avg_to_close", "avg_to_close_5d", "order_imbalance",
        "imbalance_5d", "spread_pct", "spread_5d", "range_pct", "range_5d",
        "amount_z20", "mom_5d", "mom_20d", "mom_60d", "vol_20d",
        "close_to_vwap", "illiquidity",
    ]

    output_dates = sorted(out.loc[out["date"].between(start_dt, end_dt), "date"].unique())
    if not output_dates:
        return out

    try:
        from sklearn.impute import SimpleImputer
        from sklearn.linear_model import Ridge
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
    except Exception:
        # BigQuant usually has sklearn. If not, the rule factor still returns a valid submission.
        return out

    model = None
    last_train_date = None
    rng = np.random.default_rng(RANDOM_SEED)

    for d in output_dates:
        d = pd.Timestamp(d)
        need_train = model is None or last_train_date is None or (d - last_train_date).days >= RETRAIN_EVERY_DAYS
        if need_train:
            train_start = d - pd.Timedelta(days=LOOKBACK_DAYS)
            train_mask = (
                (out["date"] < d)
                & (out["date"] >= train_start)
                & out["target_z"].notna()
            )
            train = out.loc[train_mask, feature_cols + ["target_z"]].replace([np.inf, -np.inf], np.nan)
            train = train.dropna(subset=["target_z"])

            if len(train) >= MIN_TRAIN_ROWS:
                if len(train) > MAX_TRAIN_ROWS:
                    sample_idx = rng.choice(train.index.to_numpy(), size=MAX_TRAIN_ROWS, replace=False)
                    train = train.loc[sample_idx]

                # [AI-CORE] A rolling Ridge model learns a non-linear-looking combination
                # through standardized multi-feature interactions over time. Ridge is used
                # for stability and speed under the 3-hour Notebook limit.
                model = Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                        ("ridge", Ridge(alpha=20.0, random_state=RANDOM_SEED)),
                    ]
                )
                model.fit(train[feature_cols], train["target_z"])
                last_train_date = d

        if model is not None:
            pred_mask = out["date"].eq(d)
            x_pred = out.loc[pred_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
            out.loc[pred_mask, "ai_model_pred"] = model.predict(x_pred)

    out["ai_model_pred"] = _cs_zscore(out, "ai_model_pred")
    return out


def _finalize_factor(df: pd.DataFrame, start_dt: pd.Timestamp, end_dt: pd.Timestamp) -> pd.DataFrame:
    out = df.copy()
    out["ai_available"] = out["ai_model_pred"].notna()
    out["factor"] = np.where(
        out["ai_available"],
        0.70 * out["ai_model_pred"] + 0.30 * out["rule_factor"],
        out["rule_factor"],
    )
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date")["factor"].transform(lambda s: s.fillna(s.median()))
    out["factor"] = out["factor"].fillna(0.0)
    out["factor"] = _cs_zscore(out, "factor")

    result = out.loc[out["date"].between(start_dt, end_dt), ["date", "instrument", "factor"]].copy()
    result["date"] = pd.to_datetime(result["date"]).dt.normalize()
    result["instrument"] = result["instrument"].astype(str)
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce").fillna(0.0).astype(float)
    result = result.sort_values(["date", "instrument"]).reset_index(drop=True)

    # Competition hard requirement: exactly these three columns.
    return result[["date", "instrument", "factor"]]


def main(
    data_source: Any = DEFAULT_BAR_TABLE,
    start_datetime: Any = "2025-01-01 00:00:00",
    end_datetime: Any = "2025-12-31 23:59:59",
    **kwargs: Any,
) -> pd.DataFrame:
    """Competition entry point.

    Parameters
    ----------
    data_source:
        The BigQuant platform may pass the data source name. If empty, the official
        minute-bar table `bigalpha_2026_stock_bar1m` is used.
    start_datetime, end_datetime:
        Evaluation window. The function returns factors only inside this window,
        but queries an earlier lookback window for rolling features and AI training.

    Returns
    -------
    pandas.DataFrame with exactly `date`, `instrument`, `factor`.
    """
    # Accept possible alternative argument names used by platform wrappers.
    start_datetime = kwargs.get("start_date", kwargs.get("start", start_datetime))
    end_datetime = kwargs.get("end_date", kwargs.get("end", end_datetime))

    start_dt = _to_timestamp(start_datetime, "2025-01-01 00:00:00").normalize()
    end_dt = _to_timestamp(end_datetime, "2025-12-31 23:59:59").normalize()
    if end_dt < start_dt:
        raise ValueError("end_datetime must be later than start_datetime")

    bar_table = _resolve_bar_table(data_source)
    query_start = max(pd.Timestamp("2019-01-01"), start_dt - pd.Timedelta(days=LOOKBACK_DAYS))

    daily = _load_daily_features_from_dai(bar_table, query_start, end_dt)
    features = _build_daily_features(daily)
    features = _add_rule_factor(features)
    features = _add_walk_forward_ai_factor(features, start_dt, end_dt)
    result = _finalize_factor(features, start_dt, end_dt)

    # Defensive validation before returning to the evaluator.
    if list(result.columns) != ["date", "instrument", "factor"]:
        raise ValueError("Submission output must contain exactly date, instrument, factor")
    if result["factor"].isna().any():
        raise ValueError("Submission output contains missing factor values")
    return result



## 平台测试示例

在 BigQuant AIStudio 中可以先运行：

```python
res = main("bigalpha_2026_stock_bar1m", "2025-01-01 00:00:00", "2025-01-31 23:59:59")
res.head(), res.shape, res.columns.tolist()
```

正式提交时，平台会自动 import 并调用 `main`，不要在提交版里写外部网络请求。
